# Notebook 05 — Purged Anchored Walk-Forward Design

## Purpose

Freeze the primary model-selection folds **before** Optuna and algorithm
comparison.

The input population is the Stage 04 primary long candidate set generated by
the new confirmation-gated ZigZag.

## Validation design

- Train-only calendar.
- Five anchored walk-forward folds.
- Calendar-date boundaries, not random row splits.
- First validation block starts after 50% of train-only trading dates.
- Remaining trading dates are split into five contiguous validation blocks.
- 30-trading-day conservative gap immediately before every validation block.
- Event-end overlap purging.
- All events from the same start date remain on the same temporal side.
- No model fitting.
- No unseen-test access.


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
import yaml

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


Python: 3.12.2
pandas: 2.2.3
numpy: 1.26.4


## 1. Locate repository and import validation modules

In [2]:
def locate_repository_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "configs").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Repository root was not found. Open the project root in VS Code."
    )


REPOSITORY_ROOT = locate_repository_root(Path.cwd())

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.utils.paths import (
    data_paths,
    repository_result_paths,
    resolve_data_root,
)
from src.utils.reproducibility import (
    git_commit_sha,
    software_manifest,
    stable_object_hash,
)
from src.validation.folds import (
    build_anchored_calendar_folds,
    folds_to_frame,
    normalize_trading_calendar,
)
from src.validation.purged_walk_forward import (
    audit_anchored_training_sets,
    audit_fold_integrity,
    audit_validation_window_overlap,
    fold_membership_masks,
    normalize_candidate_event_panel,
    summarize_fold,
)

print("Repository root:", REPOSITORY_ROOT)


Repository root: E:\Iran_stock_trade\financial-ml-decision-support


## 2. Load frozen Stage 04 and Stage 05 policies

In [3]:
def load_yaml(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_obj:
        value = yaml.safe_load(file_obj)
    if not isinstance(value, dict):
        raise ValueError(f"Configuration must be a mapping: {path}")
    return value


paths_config = load_yaml(REPOSITORY_ROOT / "configs" / "paths.yaml")
validation_config = load_yaml(
    REPOSITORY_ROOT / "configs" / "validation.yaml"
)

DATA_ROOT = resolve_data_root(paths_config, REPOSITORY_ROOT)
DATA_PATHS = data_paths(DATA_ROOT, paths_config)
RESULT_PATHS = repository_result_paths(
    REPOSITORY_ROOT,
    paths_config,
)

CANDIDATE_TRAIN_PATH = DATA_PATHS["candidate_data"] / "train"
LABELED_TRAIN_PATH = DATA_PATHS["labeled_data"] / "train"

stage04_policy_path = (
    RESULT_PATHS["manifests"] / "04_candidate_filter_policy.json"
)
approved_features_path = (
    RESULT_PATHS["manifests"] / "04_approved_model_features.csv"
)
frozen_universe_path = (
    RESULT_PATHS["manifests"] / "02_frozen_universe.csv"
)

for required_path in [
    stage04_policy_path,
    approved_features_path,
    frozen_universe_path,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required frozen-stage artifact is missing: {required_path}"
        )

stage04_policy = json.loads(
    stage04_policy_path.read_text(encoding="utf-8")
)
primary_validation = validation_config["primary_validation"]

assert validation_config["status"] == "stage_05_configured"
assert stage04_policy["primary_side"] == "long"
assert np.isclose(
    float(stage04_policy["primary_threshold_fraction"]),
    0.15,
)
assert np.isclose(
    float(
        primary_validation[
            "candidate_filter_threshold_fraction"
        ]
    ),
    0.15,
)
assert primary_validation["candidate_scope"] == (
    "stage_04_primary_long_candidates"
)
assert primary_validation["unseen_test_used_for_fold_design"] is False
assert primary_validation["fold_boundaries_frozen_before_model_selection"] is True

NUMBER_OF_FOLDS = int(primary_validation["number_of_folds"])
FIRST_VALIDATION_START_FRACTION = float(
    primary_validation["first_validation_start_fraction"]
)
EMBARGO_TRADING_DAYS = int(
    primary_validation["embargo_trading_days"]
)

print("Candidate path:", CANDIDATE_TRAIN_PATH)
print("Calendar source:", LABELED_TRAIN_PATH)
print("Number of folds:", NUMBER_OF_FOLDS)
print(
    "First validation start fraction:",
    FIRST_VALIDATION_START_FRACTION,
)
print("Embargo trading days:", EMBARGO_TRADING_DAYS)
print("Unseen-test used for fold design: False")


Candidate path: E:\Iran_stock_trade\financial-ml-decision-support\data_ready\candidates\train
Calendar source: E:\Iran_stock_trade\financial-ml-decision-support\data_ready\labeled\train
Number of folds: 5
First validation start fraction: 0.5
Embargo trading days: 30
Unseen-test used for fold design: False


## 3. Validate frozen universe and Stage 04 candidate files

In [4]:
frozen_universe_df = pd.read_csv(
    frozen_universe_path,
    low_memory=False,
)
frozen_symbols = sorted(
    frozen_universe_df["symbol"].astype(str).tolist()
)
frozen_symbol_set = set(frozen_symbols)

candidate_files = sorted(
    CANDIDATE_TRAIN_PATH.glob("*_train_candidates.csv")
)
labeled_train_files = sorted(
    LABELED_TRAIN_PATH.glob("*_train_labeled.csv")
)


def candidate_symbol_from_path(path: Path) -> str:
    suffix = "_train_candidates.csv"
    if not path.name.endswith(suffix):
        raise ValueError(f"Unexpected candidate file: {path.name}")
    return path.name[:-len(suffix)]


def labeled_symbol_from_path(path: Path) -> str:
    suffix = "_train_labeled.csv"
    if not path.name.endswith(suffix):
        raise ValueError(f"Unexpected labeled file: {path.name}")
    return path.name[:-len(suffix)]


candidate_file_map = {
    candidate_symbol_from_path(path): path
    for path in candidate_files
}
labeled_file_map = {
    labeled_symbol_from_path(path): path
    for path in labeled_train_files
}

assert set(candidate_file_map) == frozen_symbol_set
assert set(labeled_file_map) == frozen_symbol_set
assert len(candidate_file_map) == len(frozen_symbols)
assert len(labeled_file_map) == len(frozen_symbols)

print("Frozen symbols:", len(frozen_symbols))
print("Candidate files:", len(candidate_file_map))
print("Labeled train files:", len(labeled_file_map))


Frozen symbols: 499
Candidate files: 499
Labeled train files: 499


## 4. Build the train-only trading calendar

The market calendar is built from frozen **labeled train** dates, not candidate
dates. This makes the 30-day gap a trading-calendar gap even on dates with no
candidate events.


In [5]:
calendar_date_parts = []
calendar_error_rows = []
started = time.perf_counter()

for file_number, (symbol, path) in enumerate(
    sorted(labeled_file_map.items()),
    start=1,
):
    try:
        dates = pd.read_csv(
            path,
            usecols=["dEven"],
            low_memory=False,
        )["dEven"]
        calendar_date_parts.append(dates)
    except Exception as exc:
        calendar_error_rows.append(
            {
                "symbol": symbol,
                "file_name": path.name,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

    if (
        file_number == 1
        or file_number % 50 == 0
        or file_number == len(labeled_file_map)
    ):
        elapsed = time.perf_counter() - started
        print(
            f"[calendar] "
            f"[{file_number:>4}/{len(labeled_file_map)}] "
            f"elapsed={elapsed:,.1f}s "
            f"errors={len(calendar_error_rows)}"
        )

calendar_errors_df = pd.DataFrame(
    calendar_error_rows,
    columns=[
        "symbol",
        "file_name",
        "error_type",
        "error_message",
    ],
)

all_calendar_dates = pd.concat(
    calendar_date_parts,
    ignore_index=True,
)
trading_calendar = normalize_trading_calendar(
    all_calendar_dates.tolist()
)

calendar_summary_df = pd.DataFrame(
    [
        {
            "calendar_source": "frozen_labeled_train",
            "unique_trading_dates": len(trading_calendar),
            "first_trading_date": trading_calendar.min(),
            "last_trading_date": trading_calendar.max(),
            "unseen_test_dates_used": False,
        }
    ]
)

calendar_errors_df.to_csv(
    RESULT_PATHS["audits"] / "05_calendar_errors.csv",
    index=False,
    encoding="utf-8-sig",
)
calendar_summary_df.to_csv(
    RESULT_PATHS["audits"] / "05_candidate_calendar_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

display(calendar_summary_df)


[calendar] [   1/499] elapsed=0.1s errors=0
[calendar] [  50/499] elapsed=1.8s errors=0
[calendar] [ 100/499] elapsed=3.3s errors=0
[calendar] [ 150/499] elapsed=4.6s errors=0
[calendar] [ 200/499] elapsed=5.3s errors=0
[calendar] [ 250/499] elapsed=6.2s errors=0
[calendar] [ 300/499] elapsed=7.0s errors=0
[calendar] [ 350/499] elapsed=7.7s errors=0
[calendar] [ 400/499] elapsed=8.5s errors=0
[calendar] [ 450/499] elapsed=9.3s errors=0
[calendar] [ 499/499] elapsed=10.1s errors=0


,calendar_source,unique_trading_dates,first_trading_date,last_trading_date,unseen_test_dates_used
0,frozen_labeled_train,1690,2014-03-19,2021-03-17,False


## 5. Build the Stage 04 candidate-event panel

In [6]:
candidate_parts = []
candidate_error_rows = []
started = time.perf_counter()

usecols = [
    "dEven",
    "event_end_date",
    "meta_label",
]

for file_number, (symbol, path) in enumerate(
    sorted(candidate_file_map.items()),
    start=1,
):
    try:
        frame = pd.read_csv(
            path,
            usecols=usecols,
            low_memory=False,
        )
        frame.insert(0, "symbol", symbol)

        event_dates = pd.to_datetime(
            frame["dEven"],
            errors="coerce",
        )
        frame.insert(
            0,
            "event_id",
            (
                symbol
                + "::"
                + event_dates.dt.strftime("%Y-%m-%d")
            ),
        )
        candidate_parts.append(frame)

    except Exception as exc:
        candidate_error_rows.append(
            {
                "symbol": symbol,
                "file_name": path.name,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

    if (
        file_number == 1
        or file_number % 50 == 0
        or file_number == len(candidate_file_map)
    ):
        elapsed = time.perf_counter() - started
        print(
            f"[candidate panel] "
            f"[{file_number:>4}/{len(candidate_file_map)}] "
            f"elapsed={elapsed:,.1f}s "
            f"errors={len(candidate_error_rows)}"
        )

candidate_errors_df = pd.DataFrame(
    candidate_error_rows,
    columns=[
        "symbol",
        "file_name",
        "error_type",
        "error_message",
    ],
)

candidate_panel = normalize_candidate_event_panel(
    pd.concat(candidate_parts, ignore_index=True)
)

candidate_panel_summary_df = pd.DataFrame(
    [
        {
            "candidate_events": len(candidate_panel),
            "candidate_symbols": candidate_panel["symbol"].nunique(),
            "first_event_start_date": candidate_panel["dEven"].min(),
            "last_event_start_date": candidate_panel["dEven"].max(),
            "positive_meta_labels": int(
                candidate_panel["meta_label"].eq(1).sum()
            ),
            "negative_meta_labels": int(
                candidate_panel["meta_label"].eq(0).sum()
            ),
            "positive_meta_label_fraction": float(
                candidate_panel["meta_label"].eq(1).mean()
            ),
            "duplicate_event_ids": int(
                candidate_panel["event_id"].duplicated().sum()
            ),
        }
    ]
)

candidate_errors_df.to_csv(
    RESULT_PATHS["audits"] / "05_candidate_panel_errors.csv",
    index=False,
    encoding="utf-8-sig",
)
candidate_panel_summary_df.to_csv(
    RESULT_PATHS["audits"] / "05_candidate_panel_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

display(candidate_panel_summary_df)


[candidate panel] [   1/499] elapsed=0.1s errors=0
[candidate panel] [  50/499] elapsed=2.9s errors=0
[candidate panel] [ 100/499] elapsed=6.0s errors=0
[candidate panel] [ 150/499] elapsed=8.5s errors=0
[candidate panel] [ 200/499] elapsed=10.2s errors=0
[candidate panel] [ 250/499] elapsed=12.0s errors=0
[candidate panel] [ 300/499] elapsed=13.6s errors=0
[candidate panel] [ 350/499] elapsed=15.3s errors=0
[candidate panel] [ 400/499] elapsed=16.9s errors=0
[candidate panel] [ 450/499] elapsed=18.6s errors=0
[candidate panel] [ 499/499] elapsed=20.3s errors=0


,candidate_events,candidate_symbols,first_event_start_date,last_event_start_date,positive_meta_labels,negative_meta_labels,positive_meta_label_fraction,duplicate_event_ids
0,118464,499,2014-05-06,2021-03-14,63873,54591,0.539176,0


## 6. Freeze five anchored calendar folds

In [7]:
folds = build_anchored_calendar_folds(
    trading_calendar,
    number_of_folds=NUMBER_OF_FOLDS,
    first_validation_start_fraction=(
        FIRST_VALIDATION_START_FRACTION
    ),
    embargo_trading_days=EMBARGO_TRADING_DAYS,
)

fold_boundaries_df = folds_to_frame(folds)

fold_boundaries_path = (
    RESULT_PATHS["folds"]
    / "05_purged_anchored_walk_forward_boundaries.csv"
)
fold_boundaries_df.to_csv(
    fold_boundaries_path,
    index=False,
    encoding="utf-8-sig",
)

display(fold_boundaries_df)


,fold_id,calendar_start_date,train_end_date,embargo_start_date,embargo_end_date,validation_start_date,validation_end_date,train_end_calendar_index,validation_start_calendar_index,validation_end_calendar_index,embargo_trading_days
0,1,2014-03-19,2017-08-06,2017-08-07,2017-09-18,2017-09-19,2018-06-03,814,845,1013,30
1,2,2014-03-19,2018-04-21,2018-04-22,2018-06-03,2018-06-09,2019-02-10,983,1014,1182,30
2,3,2014-03-19,2018-12-29,2018-12-30,2019-02-10,2019-02-12,2019-10-30,1152,1183,1351,30
3,4,2014-03-19,2019-09-15,2019-09-16,2019-10-30,2019-11-02,2020-07-13,1321,1352,1520,30
4,5,2014-03-19,2020-05-30,2020-05-31,2020-07-13,2020-07-14,2021-03-17,1490,1521,1689,30


## 7. Apply purge and embargo controls without fitting any model

For each fold:

1. the historical candidate population is restricted to starts on or before the
   pre-embargo training end;
2. any retained historical event with `event_end_date >= validation_start_date`
   is purged;
3. validation consists only of candidate events whose start dates fall inside
   that fold's contiguous validation window.


In [8]:
fold_summary_rows = []
fold_integrity_rows = []
validation_assignment_parts = []

for fold in folds:
    masks = fold_membership_masks(candidate_panel, fold)

    fold_summary_rows.append(
        summarize_fold(
            candidate_panel,
            fold,
            masks,
        )
    )
    fold_integrity_rows.append(
        audit_fold_integrity(
            candidate_panel,
            fold,
            masks,
        )
    )

    validation_events = candidate_panel.loc[
        masks.validation,
        [
            "event_id",
            "symbol",
            "dEven",
            "event_end_date",
            "meta_label",
        ],
    ].copy()
    validation_events.insert(0, "fold_id", fold.fold_id)
    validation_assignment_parts.append(validation_events)

fold_summary_df = pd.DataFrame(fold_summary_rows)
fold_integrity_df = pd.DataFrame(fold_integrity_rows)
validation_assignments_df = pd.concat(
    validation_assignment_parts,
    ignore_index=True,
)

anchored_audit_df = audit_anchored_training_sets(
    candidate_panel,
    folds,
)
validation_overlap_audit_df = audit_validation_window_overlap(
    folds
)

fold_summary_df.to_csv(
    RESULT_PATHS["folds"]
    / "05_purged_anchored_walk_forward_folds.csv",
    index=False,
    encoding="utf-8-sig",
)
validation_assignments_df.to_csv(
    RESULT_PATHS["folds"]
    / "05_validation_event_assignments.csv",
    index=False,
    encoding="utf-8-sig",
)
fold_integrity_df.to_csv(
    RESULT_PATHS["audits"] / "05_fold_integrity_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
anchored_audit_df.to_csv(
    RESULT_PATHS["audits"] / "05_anchored_training_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
validation_overlap_audit_df.to_csv(
    RESULT_PATHS["audits"] / "05_validation_window_overlap_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

display(fold_summary_df)
display(fold_integrity_df)
display(anchored_audit_df)


,fold_id,train_start_date,train_end_date,embargo_start_date,embargo_end_date,validation_start_date,validation_end_date,historical_events_before_embargo,purged_event_overlap_count,embargo_candidate_event_count,...,train_negative_labels,train_positive_fraction,validation_events,validation_positive_labels,validation_negative_labels,validation_positive_fraction,train_symbols,validation_symbols,train_unique_event_start_dates,validation_unique_event_start_dates
0,1,2014-05-06,2017-08-06,2017-08-07,2017-09-18,2017-09-19,2018-06-03,63238,521,3386,...,31216,0.502272,19417,8141,11276,0.419272,443,446,786,169
1,2,2014-05-06,2018-04-21,2018-04-22,2018-06-03,2018-06-09,2019-02-10,82746,473,3295,...,43563,0.470507,13537,9522,4015,0.703405,484,473,955,167
2,3,2014-05-06,2018-12-29,2018-12-30,2019-02-10,2019-02-12,2019-10-30,96416,98,3162,...,47819,0.503530,6496,5927,569,0.912408,496,416,1122,169
3,4,2014-05-06,2019-09-15,2019-09-16,2019-10-30,2019-11-02,2020-07-13,105209,9,865,...,49116,0.533118,3145,2932,213,0.932273,498,359,1291,167
4,5,2014-05-06,2020-05-30,2020-05-31,2020-07-13,2020-07-14,2021-03-17,108794,7,425,...,49533,0.544679,9245,4301,4944,0.465224,499,427,1454,166


,fold_id,train_event_ids_in_validation,train_validation_event_start_date_overlap,train_start_after_train_end,train_event_end_on_or_after_validation_start,validation_before_validation_start,validation_after_validation_end,embargo_events_in_train,purged_events_in_train,train_has_both_classes,validation_has_both_classes
0,1,0,0,0,0,0,0,0,0,True,True
1,2,0,0,0,0,0,0,0,0,True,True
2,3,0,0,0,0,0,0,0,0,True,True
3,4,0,0,0,0,0,0,0,0,True,True
4,5,0,0,0,0,0,0,0,0,True,True


,previous_fold_id,current_fold_id,anchored_subset_ok,previous_train_events_missing_from_current,current_train_events
0,NaN,1,True,0,62717
1,1.0,2,True,0,82273
2,2.0,3,True,0,96318
3,3.0,4,True,0,105200
4,4.0,5,True,0,108787


## 8. Fold class-balance and calendar coverage audit

In [9]:
class_balance_rows = []

for row in fold_summary_df.itertuples(index=False):
    class_balance_rows.extend(
        [
            {
                "fold_id": row.fold_id,
                "partition": "train",
                "events": row.train_events,
                "positive_labels": row.train_positive_labels,
                "negative_labels": row.train_negative_labels,
                "positive_fraction": row.train_positive_fraction,
            },
            {
                "fold_id": row.fold_id,
                "partition": "validation",
                "events": row.validation_events,
                "positive_labels": row.validation_positive_labels,
                "negative_labels": row.validation_negative_labels,
                "positive_fraction": row.validation_positive_fraction,
            },
        ]
    )

fold_class_balance_df = pd.DataFrame(class_balance_rows)

fold_class_balance_df.to_csv(
    RESULT_PATHS["audits"] / "05_fold_class_balance.csv",
    index=False,
    encoding="utf-8-sig",
)

display(fold_class_balance_df)


,fold_id,partition,events,positive_labels,negative_labels,positive_fraction
0,1,train,62717,31501,31216,0.502272
1,1,validation,19417,8141,11276,0.419272
2,2,train,82273,38710,43563,0.470507
3,2,validation,13537,9522,4015,0.703405
4,3,train,96318,48499,47819,0.503530
5,3,validation,6496,5927,569,0.912408
6,4,train,105200,56084,49116,0.533118
7,4,validation,3145,2932,213,0.932273
8,5,train,108787,59254,49533,0.544679
9,5,validation,9245,4301,4944,0.465224


## 9. Freeze Stage 05 manifest

In [10]:
fold_records = json.loads(
    fold_boundaries_df.to_json(
        orient="records",
        date_format="iso",
    )
)

manifest = {
    "stage": 5,
    "status": "frozen_after_integrity_checks",
    "notebook": "05_purged_walk_forward_design.ipynb",
    "git_commit_sha": git_commit_sha(REPOSITORY_ROOT),
    "software": software_manifest(),
    "candidate_population": {
        "source": "Stage 04 primary long candidates",
        "events": len(candidate_panel),
        "symbols": int(candidate_panel["symbol"].nunique()),
        "primary_zigzag_threshold_fraction": 0.15,
        "target_column": "meta_label",
    },
    "calendar": {
        "source": "frozen_labeled_train",
        "unique_trading_dates": len(trading_calendar),
        "first_date": str(trading_calendar.min().date()),
        "last_date": str(trading_calendar.max().date()),
    },
    "primary_validation": primary_validation,
    "folds": fold_records,
    "unseen_test_used": False,
    "configuration_hash": stable_object_hash(
        {
            "validation": validation_config,
            "stage04_candidate_policy": stage04_policy,
        }
    ),
}

manifest_path = (
    RESULT_PATHS["manifests"]
    / "05_purged_anchored_walk_forward_manifest.json"
)
manifest_path.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("Manifest:", manifest_path)


Manifest: E:\Iran_stock_trade\financial-ml-decision-support\results\manifests\05_purged_anchored_walk_forward_manifest.json


## 10. Final integrity checks

In [11]:
assert calendar_errors_df.empty, (
    "Calendar construction errors exist. "
    "Review 05_calendar_errors.csv"
)
assert candidate_errors_df.empty, (
    "Candidate panel errors exist. "
    "Review 05_candidate_panel_errors.csv"
)
assert len(folds) == NUMBER_OF_FOLDS
assert len(candidate_panel) > 0
assert candidate_panel["event_id"].is_unique
assert candidate_panel["meta_label"].isin([0, 1]).all()

assert fold_integrity_df[
    "train_event_ids_in_validation"
].eq(0).all()

assert fold_integrity_df[
    "train_validation_event_start_date_overlap"
].eq(0).all()

assert fold_integrity_df[
    "train_start_after_train_end"
].eq(0).all()

assert fold_integrity_df[
    "train_event_end_on_or_after_validation_start"
].eq(0).all()

assert fold_integrity_df[
    "validation_before_validation_start"
].eq(0).all()

assert fold_integrity_df[
    "validation_after_validation_end"
].eq(0).all()

assert fold_integrity_df[
    "embargo_events_in_train"
].eq(0).all()

assert fold_integrity_df[
    "purged_events_in_train"
].eq(0).all()

assert fold_integrity_df[
    "train_has_both_classes"
].all()

assert fold_integrity_df[
    "validation_has_both_classes"
].all()

assert anchored_audit_df[
    "anchored_subset_ok"
].all()

assert not validation_overlap_audit_df[
    "validation_windows_overlap"
].any()

assert validation_assignments_df["event_id"].is_unique
assert validation_assignments_df["fold_id"].between(
    1,
    NUMBER_OF_FOLDS,
).all()

print("Notebook 05 integrity checks: PASSED")
print("Candidate long events:", len(candidate_panel))
print("Walk-forward folds:", len(folds))
print("Embargo gap:", EMBARGO_TRADING_DAYS, "trading days")
print("Purge method: event_end_overlap")
print(
    "Train/validation event overlap:",
    int(
        fold_integrity_df[
            "train_event_ids_in_validation"
        ].sum()
    ),
)
print(
    "Training label intervals reaching validation:",
    int(
        fold_integrity_df[
            "train_event_end_on_or_after_validation_start"
        ].sum()
    ),
)
print(
    "Validation window overlap pairs:",
    int(
        validation_overlap_audit_df[
            "validation_windows_overlap"
        ].sum()
    ),
)
print(
    "Anchored training subset failures:",
    int(
        (~anchored_audit_df["anchored_subset_ok"]).sum()
    ),
)
print("Unseen-test used in Stage 05 decisions: False")
print(
    "Next stage: Notebook 06 — Optuna model selection "
    "with frozen purged walk-forward folds."
)


Notebook 05 integrity checks: PASSED
Candidate long events: 118464
Walk-forward folds: 5
Embargo gap: 30 trading days
Purge method: event_end_overlap
Train/validation event overlap: 0
Training label intervals reaching validation: 0
Validation window overlap pairs: 0
Anchored training subset failures: 0
Unseen-test used in Stage 05 decisions: False
Next stage: Notebook 06 — Optuna model selection with frozen purged walk-forward folds.


## Review before freezing Stage 05

Inspect:

- `results/folds/05_purged_anchored_walk_forward_boundaries.csv`
- `results/folds/05_purged_anchored_walk_forward_folds.csv`
- `results/folds/05_validation_event_assignments.csv`
- `results/audits/05_fold_integrity_audit.csv`
- `results/audits/05_anchored_training_audit.csv`
- `results/audits/05_validation_window_overlap_audit.csv`
- `results/audits/05_fold_class_balance.csv`
- `results/audits/05_candidate_calendar_audit.csv`
- `results/manifests/05_purged_anchored_walk_forward_manifest.json`

Notebook 06 must consume these frozen fold boundaries. It must not redesign the
calendar split after seeing RF, XGBoost, Dummy, or Logistic Regression results.
